# 01. Clean `ami_meter`: physical-plausibility checks

This notebook checks for row-level physical plausibility of the delivered
`V`/`P_kw`/`Q_kvar`/`S_kva`/`power_factor`/`current_a` values.

* **HARD flags**: Physically impossible or internally inconsistent:
- negative current, 
- voltage <= 0V or > 300V, 
- |power factor| > 1,
- duplicate/missing key fields. 
These rows are **dropped**.

* **SOFT flags**: Anomalous but not confidently a fault. 
  This covers two different situations: 
  - (a) extreme relative to a circuit's own history but
  not physically impossible. An unusually high per-circuit power reading,
  which could easily be a genuine PV export or EV-charging event
  - and (b) `apparent_power_inconsistent` (a stored `S_kva` that disagrees with V x I for the same row)
  Both kinds are **kept**, just tagged with a boolean column.

Output: `ami_meter_clean`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd

from bms_sa_review.ami_analysis.lib import ami_clean as Clean
from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

# Same local Parquet store `synthetic_ami_creation` writes to (`Config.STORE_DIR`)
# -- addressed the same way as `ami_raw`/`ami_meter`/`ami_raw_phaseseparate`,
# rather than this notebook's own separate env-var lookup, so this table stays
# addressable the same way as the rest of the pipeline downstream (e.g. the
# disaggregation algorithms this cleaned table is ultimately meant to feed).
ARTEFACT_DIR = REPO_ROOT / "bms_sa_review" / "ami_analysis" / "artefacts"
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()


## 1. Load `ami_meter`


In [ ]:
MONTHS = con.sql(f"""
    SELECT DISTINCT year, month FROM read_parquet(
        '{Config.store_path("ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
    ORDER BY year, month
""").df()
MONTHS_LIST = list(MONTHS.itertuples(index=False, name=None))
print(f"{len(MONTHS_LIST):,} landed (year, month) partitions to process.")
MONTHS


In [ ]:
def read_month(year: int, month: int) -> pd.DataFrame:
    return con.sql(f"""
        SELECT * FROM read_parquet(
            '{Config.store_path("ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
        WHERE year = {year} AND month = {month}
    """).df()


## 2. Run every check, one month at a time

In [ ]:
CLEAN_MONTH_DIR = Config.store_path("ami_meter_clean")
CLEAN_MONTH_DIR.mkdir(parents=True, exist_ok=True)

quality_reports = []
removed_samples = []
MAX_REMOVED_SAMPLE_ROWS_PER_MONTH = 2000

for year, month in MONTHS_LIST:
    month_frame = read_month(year, month)
    if not len(month_frame):
        continue

    flags = {
        "voltage_implausible": Clean.flag_voltage_outliers(month_frame),
        "power_factor_implausible": Clean.flag_power_factor_outliers(month_frame),
        "current_negative": Clean.flag_negative_current(month_frame),
        "apparent_power_inconsistent": Clean.flag_apparent_power_inconsistency(month_frame),
        "duplicate_reading": Clean.flag_duplicate_readings(month_frame),
        "missing_critical_field": Clean.flag_missing_critical_fields(month_frame),
        "power_magnitude_extreme": Clean.flag_extreme_power_magnitude(month_frame),
    }

    report = Clean.build_quality_report(month_frame, flags)
    report.insert(0, "month", month)
    report.insert(0, "year", year)
    quality_reports.append(report)

    clean_frame, removed_frame = Clean.apply_cleaning(month_frame, flags)

    if len(removed_frame):
        removed_samples.append(removed_frame.sample(
            n=min(MAX_REMOVED_SAMPLE_ROWS_PER_MONTH, len(removed_frame)), random_state=0,
        ))

    part_dir = CLEAN_MONTH_DIR / f"dt_month={year:04d}-{month:02d}"
    part_dir.mkdir(parents=True, exist_ok=True)
    clean_frame.to_parquet(part_dir / "part.parquet", compression="zstd", index=False)

    print(f"{year}-{month:02d}: {len(month_frame):,} rows in, "
          f"{len(removed_frame):,} dropped (hard), {len(clean_frame):,} written.")
    hard_breakdown = report[report.kind == "hard"].sort_values("n_flagged", ascending=False)
    if hard_breakdown.n_flagged.sum():
        display(hard_breakdown[["flag", "n_flagged", "share_flagged"]])


## 3. Quality report across every month


In [ ]:
quality_report = pd.concat(quality_reports, ignore_index=True)
quality_report.to_csv(ARTEFACT_DIR / "phase7_ami_meter_quality_report.csv", index=False)

summary = (
    quality_report.groupby(["flag", "kind"], as_index=False)
    .agg(n_flagged=("n_flagged", "sum"))
)
display(summary)


In [ ]:
removed_sample = pd.concat(removed_samples, ignore_index=True) if removed_samples else pd.DataFrame()
removed_sample.to_csv(ARTEFACT_DIR / "phase7_ami_meter_removed_sample.csv", index=False)
print(f"Saved a {len(removed_sample):,}-row sample of dropped (hard-flagged) rows for manual review, "
      f"across {len(removed_samples)} month(s) with at least one drop.")
removed_sample.head(20)


## 4. Sanity-check the cleaned table

Re-open `ami_meter_clean` from disk (not the in-memory frames above) and
confirm row counts and that every HARD flag is now clean, while
`power_magnitude_extreme` (soft) rows are still present as expected.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_clean AS
    SELECT * FROM read_parquet(
        '{CLEAN_MONTH_DIR.as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

before_after = con.sql(f"""
    SELECT
      (SELECT count(*) FROM read_parquet('{Config.store_path("ami_meter").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)) AS n_rows_before,
      (SELECT count(*) FROM ami_meter_clean) AS n_rows_after,
      (SELECT count(*) FROM ami_meter_clean WHERE power_magnitude_extreme) AS n_power_magnitude_extreme_kept,
      (SELECT count(*) FROM ami_meter_clean WHERE apparent_power_inconsistent) AS n_apparent_power_inconsistent_kept
""").df()
before_after


In [ ]:
recheck = con.sql("""
    SELECT
      sum(CASE WHEN V <= 0 OR V > 300 THEN 1 ELSE 0 END) AS n_voltage_implausible,
      sum(CASE WHEN abs(power_factor) > 1.001 THEN 1 ELSE 0 END) AS n_pf_implausible,
      sum(CASE WHEN current_a < 0 THEN 1 ELSE 0 END) AS n_current_negative
    FROM ami_meter_clean
""").df()
assert (recheck.iloc[0] == 0).all(), "a hard-flagged condition survived into ami_meter_clean -- investigate before proceeding"
print("Confirmed: no hard-flagged rows remain in ami_meter_clean.")
recheck


## 5. Is `apparent_power_inconsistent` device-specific?


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_circuit_metadata AS
    SELECT * FROM read_parquet('{Config.store_path("ami_circuit_metadata").as_posix()}')
""")

by_device_type = con.sql("""
    SELECT
      m.device_type,
      count(*) AS n_rows,
      sum(CASE WHEN c.apparent_power_inconsistent THEN 1 ELSE 0 END) AS n_flagged,
      sum(CASE WHEN c.apparent_power_inconsistent THEN 1 ELSE 0 END) * 1.0 / count(*) AS share_flagged
    FROM ami_meter_clean c
    LEFT JOIN ami_circuit_metadata m USING (circuit_id)
    GROUP BY m.device_type
    ORDER BY share_flagged DESC
""").df()
by_device_type


In [ ]:
by_circuit = con.sql("""
    SELECT circuit_id, count(*) AS n_rows,
           sum(CASE WHEN apparent_power_inconsistent THEN 1 ELSE 0 END) AS n_flagged,
           sum(CASE WHEN apparent_power_inconsistent THEN 1 ELSE 0 END) * 1.0 / count(*) AS share_flagged
    FROM ami_meter_clean
    GROUP BY circuit_id
    HAVING count(*) > 100
    ORDER BY share_flagged DESC
    LIMIT 20
""").df()
by_circuit
